# Image OCR + Local RAG

**How to use:**
1. Edit `IMAGE_PATH` and/or `QUESTION` in the cell below
2. Run All cells (menu: *Run* → *Run All Cells*)

Leave `IMAGE_PATH = ""` to skip adding a new image (just ask).
Leave `QUESTION = ""` to skip asking (just add an image).

**One-time setup** (run in a terminal):
```
py -m pip install -r requirements.txt
ollama pull qwen2.5:1.5b
ollama pull nomic-embed-text
```
Plus install the Tesseract OCR binary: https://github.com/UB-Mannheim/tesseract/wiki

## Förslag på frågor att prova (Station 1)

Kör en bild först (t.ex. första sidan i en bok, ett kvitto eller en Wikipedia-skärmdump om en känd person). Prova sedan dessa frågor i tur och ordning:

1. **Faktisk fråga från texten** — t.ex. *"Var utspelar sig handlingen?"* eller *"Vad var totalsumman?"*
   → Modellen ska svara grundat i den OCR-extraherade kontexten.

2. **Fråga vars svar INTE finns i texten, men finns i modellens träningsdata** — t.ex. *"När föddes författaren?"* eller *"Vilket år publicerades boken?"*
   → Säger modellen *"finns inte i kontexten"* (bra grounding)? Eller läcker den fakta från träningen (teaching moment)?

3. **Skärp grounding-prompten** — i cellen *Step 2* nedan, ändra `system`-meddelandet till:
   > *"Svara endast om svaret ordagrant finns i kontexten. Citera den exakta meningen som stödjer ditt svar. Annars svara: 'finns inte i kontexten'."*

   Kör båda frågorna igen. Vad ändras?

In [ ]:
# === EDIT ME ===
IMAGE_PATH = "receipt.png"       # eller "unknown-book.png"
QUESTION   = "vad handlar texten om?"

## Setup (imports + tiny JSON-backed knowledge base)

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import ollama
import pytesseract
from PIL import Image

# Auto-detect Tesseract — funkar för både "for all users" och "just for me"-install
for _p in [
    shutil.which("tesseract"),
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    str(Path.home() / r"AppData\Local\Programs\Tesseract-OCR\tesseract.exe"),
]:
    if _p and Path(_p).exists():
        pytesseract.pytesseract.tesseract_cmd = _p
        break

DB_PATH     = Path("rag_db.json")
LLM_MODEL   = "qwen2.5:1.5b"
EMBED_MODEL = "nomic-embed-text"
TOP_K       = 3

def embed(text):
    return ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"]

def load_db():
    try:
        return json.loads(DB_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, FileNotFoundError):
        return {"docs": []}

def save_db(db):
    DB_PATH.write_text(json.dumps(db, indent=2), encoding="utf-8")

## Step 1 — OCR the image and add it to the knowledge base

In [ ]:
if IMAGE_PATH:
    text = pytesseract.image_to_string(Image.open(IMAGE_PATH)).strip()
    if text:
        db = load_db()
        db["docs"].append({"path": IMAGE_PATH, "text": text, "embedding": embed(text)})
        save_db(db)
        print(f"Added {IMAGE_PATH}")
        print(f"--- extracted ---\n{text}\n-----------------")
    else:
        print(f"No text extracted from {IMAGE_PATH}")
else:
    print("(skipping add — IMAGE_PATH is empty)")

## Step 2 — Retrieve top-k chunks and ask the local LLM

In [ ]:
if QUESTION:
    db = load_db()
    if not db["docs"]:
        print("Knowledge base is empty. Add an image first.")
    else:
        q_emb = np.array(embed(QUESTION))
        embs  = np.array([d["embedding"] for d in db["docs"]])
        scores = embs @ q_emb / (np.linalg.norm(embs, axis=1) * np.linalg.norm(q_emb))
        top = np.argsort(scores)[-TOP_K:][::-1]
        context = "\n\n---\n\n".join(db["docs"][i]["text"] for i in top)

        response = ollama.chat(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Answer the user's question with the help of the provided context."},
                {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {QUESTION}"},
            ],
        )
        print(response["message"]["content"])
else:
    print("(skipping ask — QUESTION is empty)")

### Diskussion

- Vilket av svaren ovan kom **från texten** och vilket kom **från modellens minne** (träningsdata)?
- Hur skulle ni märka skillnaden i en produktionsapp där användaren *inte* ser kontexten?
- Vad är priset för "strikt grounding" (citatkrav)? Vilka frågor blir svårare att svara på?
- Vad händer om kontexten själv innehåller en motsägelse mot träningsdatan? Vad ska modellen lita på?

## (Optional) See what's in the knowledge base

In [ ]:
db = load_db()
if not db["docs"]:
    print("Empty.")
else:
    for i, d in enumerate(db["docs"]):
        preview = d["text"][:80].replace("\n", " ")
        print(f"[{i}] {d['path']} — {preview}...")